In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from waymo_agent import *

In [3]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [4]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *

#### Load ENV

In [5]:
envconfig = EnvConfig()
cfg = envconfig

In [19]:
env = RideShareEnv(cfg)
G = env.graph

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [20]:
veh = env.observation_curr["vehicles"]
veh.head(2)

,vehicle_id,loc_x_norm,loc_y_norm,battery,status,ride_id
0,0,0.0439,0.0849,0.832625,0,-1
1,1,-0.1379,0.0819,0.659575,0,-1


In [21]:
act_rides = env.observation_curr["active_rides"]
act_rides.sample(2)

,ride_id,vehicle_id,pickup_node,pickup_x_norm,pickup_y_norm,dropoff_node,dropoff_x_norm,dropoff_y_norm,price,total_trip_distance_meters,trip_distance_remaining_meters,pickup_distance_remaining_meters,route_nodes,route_edge_idx,route_dist_on_edge,complete
16,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,[],0,0.0,False
15,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,[],0,0.0,False


In [22]:
requests = env.observation_curr["pending_requests"]
requests[requests.f_real_requests].tail(2)

,request_id,request_dt,pickup_node_id,pickup_x_norm,pickup_y_norm,cust_id,cust_bias,cust_temperature,dropoff_node_id,dropoff_x_norm,dropoff_y_norm,route_nodes,route_edge_idx,route_dist_on_edge,distance_meters,est_cost,price,max_wait_time,wait_time,status
0,7,2025-01-07 21:35:16.925913,42444051,0.0782,-0.0346,15996,1.125914,2.181874,1773076513,-0.1664,-0.5488,"[42444051, 11337295608, 42449685, 42449932, 42...",NaN,0.0,4088.148903,4.088149,NaN,0.25,0.0,0


In [ ]:
requests["status"] = RequestStatusEnum.ACCEPTED
requests["price"] = np.random.uniform(5, 20, size=len(requests))

In [33]:
random_dispatch_array = env.action_space["dispatch"].sample()
random_dispatch_array

array([ 4, 11, -1,  8, 11, 23,  5, -1,  4,  6, 19, -1, 21, 15, 23, 13, 18,
       14, 15,  5,  5, 23,  2, -1, 14])

In [34]:
out = env._match_dispatches({"dispatch": random_dispatch_array}, requests)

In [35]:
matches_orig = out.copy(deep=True)
matches = out.pop("vehicle_id")
r_dispatch = out

In [36]:
newly_assigned = requests.f_need_dispatch & (matches != env.config.no_action_id)
newly_assigned[0:5]

0     True
1    False
2    False
3    False
4    False
dtype: bool

In [37]:
new_status = np.select(  # update request statuses based on dispatch matches
    [newly_assigned],
    [RequestStatusEnum.ASSIGNED],
    requests.status,
)


matches_orig["newly_assigned"] = newly_assigned

In [38]:
matches_orig.head(5)

,vehicle_id,penalty_multiple_dispatch_assignment,penalty_assign_to_unavailable_vehicle,newly_assigned
0,4,0.1,-0.0,True
1,11,0.1,-0.0,False
2,-1,0.0,0.0,False
3,8,0.1,-0.0,False
4,-1,-0.1,-0.0,False


In [39]:
f_new = matches_orig["newly_assigned"]
new_rides = ActiveRideDF.from_requests(requests[f_new], matches_orig["vehicle_id"][f_new])

In [40]:
new_rides

,ride_id,vehicle_id,pickup_node,pickup_x_norm,pickup_y_norm,dropoff_node,dropoff_x_norm,dropoff_y_norm,price,est_cost,total_trip_distance_meters,trip_distance_remaining_meters,pickup_distance_remaining_meters,route_nodes,route_edge_idx,route_dist_on_edge,complete
0,7,4,42444051,0.0782,-0.0346,1773076513,-0.1664,-0.5488,14.881693,4.088149,4088.148926,4088.148926,4088.148926,"[42444051, 11337295608, 42449685, 42449932, 42...",0,0.0,False
